<h1 style="font-size: 1.6rem; font-weight: bold">ITO 5217: Natural Language Processing</h1>
<h1 style="font-size: 1.6rem; font-weight: bold">Module 5.2: Word Embeddings</h1>
<p style="margin-top: 5px; margin-bottom: 5px;">Monash University Australia</p>
<p style="margin-top: 5px; margin-bottom: 5px;">Jupyter Notebook by: Tristan Sim Yook Min</p>
References: Information Source from Monash Faculty of Information Technology

---

### **Word2vec (Mikolov et al., 2013)**

**Core idea:** Instead of counting co-occurrences directly, train a classifier on a fake prediction task, then use the learned weights as word embeddings.

The task is: given a word, predict which words are likely to appear nearby. We don't actually care about the prediction itself. What we want are the **weight matrices** that the model learns during training, because those weights *are* the word vectors.

This is **self-supervised learning**: the training signal comes from the text itself. A word that appears near "apricot" in the corpus serves as the positive label. No human annotation is needed.

**Key property:** The resulting vectors capture semantic relationships. For example, the vector arithmetic $\text{king} - \text{man} + \text{woman} \approx \text{queen}$ emerges naturally from training on enough data. Country-capital pairs, verb tenses, and other regularities all show up as consistent directions in the vector space.

<br>

### **Word2vec: Skip-gram**

**Task:** Given a center word $w_t$, predict the surrounding context words within a window of size $c$.

**Objective:** Maximize the average log probability across all positions in the corpus:

$$\frac{1}{T} \sum_{t=1}^{T} \sum_{\substack{-c \leq j \leq c, \\ j \neq 0}} \log p(w_{t+j} \mid w_t)$$

**Example** (window size $c = 2$): For the sentence "...problems turning **into** banking crises as...", with center word "into", the model tries to predict: "problems", "turning", "banking", "crises".

#### **Computing $p(w_O \mid w_I)$ with Softmax**

Each word has two vectors: $v_w$ (when it's the input/center word) and $v'_w$ (when it's the output/context word). The probability of seeing output word $w_O$ given input word $w_I$ is:

$$p(w_O \mid w_I) = \frac{\exp\left({v'_{w_O}}^\top v_{w_I}\right)}{\sum_{w=1}^{W} \exp\left({v'_w}^\top v_{w_I}\right)}$$

Where:
- The numerator computes the dot product between the output vector of $w_O$ and the input vector of $w_I$, then exponentiates it (making it strictly positive)
- The denominator sums over **all** words in the vocabulary (size $W$), normalizing the result to a value between 0 and 1 so it can be interpreted as a probability

**Practical note:** Computing the denominator is extremely expensive since it requires iterating over the entire vocabulary. Tricks like **Negative Sampling** and **Hierarchical Softmax** are used to make this tractable.

<br>

### **Word2vec: CBOW (Continuous Bag of Words)**

CBOW is the reverse of Skip-gram: given all the **context words**, predict the **center word**.

**Objective:**

$$\frac{1}{T} \sum_{t=1}^{T} \log p(w^{(t)} \mid w^{(t-c)}, \dots, w^{(t-1)}, w^{(t+1)}, \dots, w^{(t+c)})$$

The context word vectors are summed together in a projection layer, then fed through softmax to predict the center word.

| | Skip-gram | CBOW |
|---|---|---|
| Input | Center word | Context words |
| Output | Context words | Center word |
| Strengths | Better for rare words | Faster to train, good for frequent words |

<br>

### **GloVe: Global Vectors (Pennington et al., 2014)**

GloVe sits between matrix factorization methods (like LSA) and prediction-based methods (like Word2vec).

**Key difference from Word2vec:** Instead of learning from small local windows, GloVe starts from a **global word-context co-occurrence matrix** $X$, where $X(i, j)$ stores how often word $i$ appears within a window of word $j$ across the entire corpus. This matrix is computed once and stays fixed.

**Objective function:**

$$J = \sum_{i,j=1}^{W} f(X_{ij}) \left( w_i^T w_j + b_i + b_j - \log X_{ij} \right)^2$$

This is essentially a **weighted squared error** between the dot product of two word vectors (plus bias terms $b_i, b_j$) and the log of their observed co-occurrence count. The weighting function $f(X_{ij})$ caps the influence of very frequent pairs so they don't dominate the objective.

#### **Comparison on word analogy tasks (300-dim, 6B tokens)**

| Model | Semantic | Syntactic | Total |
|---|---|---|---|
| SVD | 6.3 | 8.1 | 7.3 |
| SVD-L | 56.6 | 63.0 | 60.1 |
| CBOW | 63.6 | 67.4 | 65.7 |
| Skip-gram | 73.0 | 66.0 | 69.1 |
| **GloVe** | **77.4** | 67.0 | **71.7** |

GloVe achieves the best overall accuracy, particularly on semantic analogies.

<br>

### **fastText (Bojanowski et al., 2017)**

**Problem:** Word2vec and GloVe treat each word as an atomic unit and ignore internal structure (morphology). This is a limitation for morphologically rich languages and for handling rare or unseen words.

**Solution:** Represent each word as a **bag of character n-grams** (typically 3 to 6 characters), learn a vector for each n-gram, and represent the full word as the **sum** of its n-gram vectors.

**Example:** The word "where" with 3-gram decomposition produces: `<wh`, `whe`, `her`, `ere`, `re>` plus the full token `<where>`. (The angle brackets mark word boundaries, so the trigram `her` from "where" is distinct from the standalone word `<her>`.)

**Key advantage:** fastText can produce vectors for **out-of-vocabulary (OOV) words** by summing the n-gram vectors of their subword components.

<br>

### **Evaluating Word Vectors**

There are two main approaches to evaluating the quality of word embeddings:

**Extrinsic evaluation:** Use the embeddings as input features in a downstream task (e.g., GLUE benchmark, SQuAD, NLI, text classification). This tells you whether the embeddings help in a real application, but it is expensive to run and hard to isolate the effect of the embeddings from other system components.

**Intrinsic evaluation:** Test the embeddings directly on tasks like word similarity and word analogy (e.g., "man is to woman as king is to ___"). These are fast to compute but may not predict how well the embeddings will perform on actual downstream tasks.

#### **Word Analogy (Intrinsic)**

Given a relationship a:b :: c:?, find the word whose vector best satisfies:

$$\vec{d} \approx \vec{b} - \vec{a} + \vec{c}$$

by searching for the nearest neighbor (using cosine similarity) while excluding the input words.

<br>

### **The Polysemy Problem**

**Limitation of static embeddings:** Word2vec, GloVe, and fastText all assign a **single fixed vector** to each word type, regardless of context. But words often have multiple meanings (polysemy):

- "bank" can mean a financial institution or a river bank
- "play" can mean a theatrical performance or an action in a game
- "bark" can mean a dog's sound or the outer layer of a tree

A single vector can only represent an average of all these senses, which is a lossy compromise. This motivates **contextualized embeddings**.

<br>

---

<br>

### **Contextualized Word Embeddings**

**Core idea:** Instead of one fixed vector per word type, compute a **different vector for each token occurrence** based on its surrounding sentence. The same word gets different representations depending on context.

<br>

### **ELMo (Embeddings from Language Models, 2018)**

ELMo examines the **entire sentence** before assigning an embedding to each word in it.

**Architecture:**
- A **bi-directional LSTM** (biLSTM) is trained as a language model: the forward LSTM predicts the next word, and the backward LSTM predicts the previous word
- The model has multiple layers, and ELMo **saves the representations from all layers** (not just the final one)

#### **How the ELMo representation is built**

For a word at position $t$, ELMo combines representations from every layer using learned scalar weights:

$$\text{ELMo}_t = \lambda_2 \cdot h_t^{(2)} + \lambda_1 \cdot h_t^{(1)} + \lambda_0 \cdot h_t^{(0)}$$

Where:
- $h_t^{(0)}$ = the initial token embedding (bottom layer)
- $h_t^{(1)}$ = the first biLSTM layer's hidden state for token $t$
- $h_t^{(2)}$ = the second biLSTM layer's hidden state for token $t$
- $\lambda_0, \lambda_1, \lambda_2$ = scalar weights learned during fine-tuning on the downstream task

**Why multiple layers matter:** Different layers capture different kinds of information. Lower layers tend to capture syntax (part-of-speech, word structure), while higher layers capture semantics (word sense, meaning in context).

**Results:** ELMo achieved large improvements across 6 NLP tasks including SNLI (+5.8%), NER (+21%), SQuAD (+25%), and parsing (+25%) over previous baselines.

<br>

### **BERT (Bidirectional Encoder Representations from Transformers, 2018)**

BERT follows the same philosophy as ELMo (context-dependent word representations) but replaces LSTMs with **Transformers**, which can attend to all positions simultaneously rather than processing sequentially.

**Key details:**
- Uses 12-24 layers of **bidirectional Transformer** encoders
- Pre-trained on two tasks: **Masked Language Modeling** (predict randomly masked tokens) and **Next Sentence Prediction** (predict whether two sentences are consecutive)
- A special `[CLS]` token is prepended to every input. The final hidden state of `[CLS]` serves as the aggregate sentence representation for classification tasks
- Accepts up to 512 tokens per input

**Impact:** BERT and its family of models (RoBERTa, ALBERT, DistilBERT, etc.) produced massive improvements across NLP benchmarks and dominated the field for years.

#### **ELMo vs. BERT at a glance**

| | ELMo | BERT |
|---|---|---|
| Architecture | Bi-directional LSTM | Bi-directional Transformer |
| Layers | 2 LSTM layers | 12-24 Transformer layers |
| Context | Sequential (left-to-right + right-to-left, concatenated) | Full bidirectional attention (all tokens attend to all tokens) |
| Pre-training task | Language modeling (predict next/previous word) | Masked token prediction + next sentence prediction |
| Usage | Feature extraction (frozen embeddings fed to task model) | Fine-tuning (entire model is updated for the task) |
